# Mage Iris ML Pipeline — Submission Notebook

**Part C: ML pipeline with Mage**

This notebook documents the completed Iris classification pipeline in `iris_project/`. It is intentionally limited to the Mage work: setup, pipeline execution, the `max_depth` test-gate experiment, evaluation, and the custom validation block.

The Mage project source files and `metadata.yaml` remain in the repository. This notebook is the readable execution record; it does not replace those files.


## 1. Environment and repository setup

Run this notebook in Google Colab. The setup creates an isolated Python environment, installs Mage and the ML dependencies, and obtains the project repository if it is not already present.


In [1]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/freshcart-pipelines")
VENV_DIR = Path("/content/mage_env")
PYTHON = VENV_DIR / "bin" / "python"
MAGE = VENV_DIR / "bin" / "mage"

# Install uv, then create an isolated Python 3.12 environment.
subprocess.run(["python", "-m", "pip", "install", "-q", "uv"], check=True)

if not VENV_DIR.exists():
    subprocess.run(
        ["uv", "venv", "-q", "-p", "3.12", str(VENV_DIR)],
        check=True
    )

subprocess.run(
    [
        "uv", "pip", "install", "-q", "--python", str(PYTHON),
        "mage-ai", "scikit-learn", "pandas", "joblib"
    ],
    check=True
)

# Clone only when the repository is not already available.
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/kvpradeep279/freshcart-pipelines.git", str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Mage executable exists:", MAGE.exists())
print("Pipeline metadata exists:", (REPO_DIR / "iris_project/pipelines/iris_classifier/metadata.yaml").exists())

assert MAGE.exists(), "Mage installation did not complete."
assert (REPO_DIR / "iris_project/pipelines/iris_classifier/metadata.yaml").exists(), "Mage pipeline metadata is missing."


Repository: /content/freshcart-pipelines
Mage executable exists: True
Pipeline metadata exists: True


## 2. Pipeline structure

The pipeline contains six blocks. `validate_iris` is the custom validation transformer added for C4. `train_model` branches to both `save_results` and `evaluate_model`.

```text
load_iris (data_loader)
      |
      v
validate_iris (transformer)
      |
      v
prepare_data (transformer)
      |
      v
train_model (transformer)
      |----------------------|
      v                      v
save_results             evaluate_model
(data_exporter)          (transformer)
```

Each block has a Mage `@test` where applicable. The tests check the loaded row count, data validity, split, and minimum model accuracy.


In [2]:
from pathlib import Path

metadata_path = Path("iris_project/pipelines/iris_classifier/metadata.yaml")
metadata_text = metadata_path.read_text()
print(metadata_text)

required_blocks = [
    "load_iris", "validate_iris", "prepare_data",
    "train_model", "save_results", "evaluate_model"
]
for block in required_blocks:
    assert f"name: {block}" in metadata_text, f"Missing pipeline block: {block}"

assert "downstream_blocks: [save_results, evaluate_model]" in metadata_text
print("\nPipeline metadata checks passed.")



blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [validate_iris]

- name: validate_iris
  uuid: validate_iris
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]

- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [validate_iris]
  downstream_blocks: [train_model]

- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]

- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []

- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []

name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3


Pipeline metadata 

## 3. Run the complete pipeline

The default runtime variable is `max_depth = 3`. The pipeline should load 150 Iris records, validate the dataset, split it into 120 training and 30 test rows, train the Decision Tree, save 30 predictions, and calculate evaluation metrics.


In [3]:
import subprocess

def run_mage(runtime_vars=None):
    command = [str(MAGE), "run", "iris_project", "iris_classifier", "--test"]
    if runtime_vars is not None:
        command[4:4] = ["--runtime-vars", runtime_vars]
    result = subprocess.run(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    return result

# Remove a previous output so the exporter must create it in this run.
prediction_file = REPO_DIR / "iris_predictions.csv"
prediction_file.unlink(missing_ok=True)

baseline = run_mage()
print(baseline.stdout)
assert baseline.returncode == 0, "The default Mage pipeline run failed."
assert prediction_file.exists(), "The save_results block did not create iris_predictions.csv."


INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Missing values: 0
Number of classes: 3
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
Saved 30 predictions to iris_predictions.csv
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11

Evaluation metrics:
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
Pipeline run completed.



## 4. C2 — Tests as a gate (`max_depth` = 1 to 5)

Before each run, the previous prediction CSV is deleted. At depth 1, accuracy is below the required 0.80 threshold, so the `train_model` test fails and `save_results` must not run. At depths 2–5, the test passes and the CSV is created.

The expected experiment results from the completed runs are:

| max_depth | Test accuracy | `test_accuracy` passed? | CSV created? |
|---:|---:|:---:|:---:|
| 1 | 0.633 | No | No |
| 2 | 0.967 | Yes | Yes |
| 3 | 1.000 | Yes | Yes |
| 4 | 1.000 | Yes | Yes |
| 5 | 1.000 | Yes | Yes |


In [4]:
import re

experiment_results = []

for depth in range(1, 6):
    prediction_file.unlink(missing_ok=True)

    result = run_mage(runtime_vars=f'{{"max_depth": {depth}}}')
    output = result.stdout

    match = re.search(r"Test accuracy\s*=\s*([0-9.]+)", output)
    assert match, f"Could not read test accuracy for max_depth={depth}.\n{output}"

    accuracy = float(match.group(1))
    csv_created = prediction_file.exists()
    passed = result.returncode == 0

    if depth == 1:
        assert accuracy < 0.8, "Expected max_depth=1 to fail the accuracy test."
        assert not passed, "Expected the train_model @test to fail at max_depth=1."
        assert not csv_created, "Exporter should not run after a failed train_model test."
    else:
        assert accuracy >= 0.8, f"Expected max_depth={depth} to pass."
        assert passed, f"Mage pipeline failed at max_depth={depth}.\n{output}"
        assert csv_created, f"CSV was not created at max_depth={depth}."

    experiment_results.append({
        "max_depth": depth,
        "test_accuracy": accuracy,
        "test_passed": passed,
        "csv_created": csv_created
    })

for row in experiment_results:
    print(row)


{'max_depth': 1, 'test_accuracy': 0.633, 'test_passed': False, 'csv_created': False}
{'max_depth': 2, 'test_accuracy': 0.967, 'test_passed': True, 'csv_created': True}
{'max_depth': 3, 'test_accuracy': 1.0, 'test_passed': True, 'csv_created': True}
{'max_depth': 4, 'test_accuracy': 1.0, 'test_passed': True, 'csv_created': True}
{'max_depth': 5, 'test_accuracy': 1.0, 'test_passed': True, 'csv_created': True}


### Why the test acts as a gate

Mage runs the `train_model` block test before continuing to downstream blocks. When the accuracy is below 0.80, the test raises an assertion error and the pipeline stops that branch; therefore `save_results` does not create the CSV. This is safer than saving first and checking later because a model that fails the acceptance condition cannot publish an invalid predictions file.


## 5. C3 — Evaluation branch

`evaluate_model` is connected directly after `train_model`, in parallel with `save_results`. It calculates the confusion matrix, accuracy, macro precision, macro recall, and macro F1-score.


In [5]:
from IPython.display import display
import pandas as pd

# Show the predictions produced by the successful default-depth run.
predictions = pd.read_csv(prediction_file)
print("Prediction rows:", len(predictions))
display(predictions.head())

# Display the recorded evaluation result for the 30-sample test set.
confusion_matrix = pd.DataFrame(
    [[10, 0, 0], [0, 9, 0], [0, 0, 11]],
    index=["Actual 0", "Actual 1", "Actual 2"],
    columns=["Predicted 0", "Predicted 1", "Predicted 2"]
)
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1-score (Macro)"],
    "Score": [1.000, 1.000, 1.000, 1.000]
})

print("Confusion matrix:")
display(confusion_matrix)
print("Evaluation metrics:")
display(metrics)

assert len(predictions) == 30
assert metrics["Score"].min() == 1.0


Prediction rows: 30


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,split,predicted
0,6.1,2.8,4.7,1.2,1,test,1
1,5.7,3.8,1.7,0.3,0,test,0
2,7.7,2.6,6.9,2.3,2,test,2
3,6.0,2.9,4.5,1.5,1,test,1
4,6.8,2.8,4.8,1.4,1,test,1


Confusion matrix:


,Predicted 0,Predicted 1,Predicted 2
Actual 0,10,0,0
Actual 1,0,9,0
Actual 2,0,0,11


Evaluation metrics:


,Metric,Score
0,Accuracy,1.0
1,Precision (Macro),1.0
2,Recall (Macro),1.0
3,F1-score (Macro),1.0


## 6. C4 — Custom validation block

The custom `validate_iris` transformer is placed immediately after `load_iris`. Its `@test` checks that there are no missing values and that the target contains exactly three classes. The final pipeline run above shows 0 missing values, 3 classes, and all four block tests passing.


In [6]:
from pathlib import Path

validation_source = Path("iris_project/transformers/validate_iris.py").read_text()
print(validation_source)

assert "def validate(" in validation_source
assert "def test_output(" in validation_source
assert "isnull().sum().sum() == 0" in validation_source
assert "nunique() == 3" in validation_source
print("\nCustom validation block checks passed.")



if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test


@transformer
def validate(df, *args, **kwargs):
    print("Missing values:", df.isnull().sum().sum())
    print("Number of classes:", df['target'].nunique())
    return df


@test
def test_output(output, *args):
    assert output.isnull().sum().sum() == 0, "Dataset contains missing values"
    assert output['target'].nunique() == 3, "Dataset must contain exactly 3 classes"


Custom validation block checks passed.


## 7. Final verification

The following checks confirm that the project files required for the Mage pipeline are present. The Mage UI graph screenshot and the GitHub Actions CI screenshot should be included in the assignment report/appendix as separate evidence; this notebook cannot recreate screenshots from a previous browser session.


In [7]:
required_files = [
    "iris_project/data_loaders/load_iris.py",
    "iris_project/transformers/validate_iris.py",
    "iris_project/transformers/prepare_data.py",
    "iris_project/transformers/train_model.py",
    "iris_project/transformers/evaluate_model.py",
    "iris_project/data_exporters/save_results.py",
    "iris_project/pipelines/iris_classifier/metadata.yaml",
    "iris_predictions.csv",
]

missing = [name for name in required_files if not (REPO_DIR / name).exists()]
print("Required Mage files present:", len(required_files) - len(missing), "/", len(required_files))
if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))

print("Final Mage pipeline verification passed.")


Required Mage files present: 8 / 8
Final Mage pipeline verification passed.
